In [1]:
# i had tried using flwr[simulation] however was very much unsuccessful (some SQlite Error) leading to the creation of this file.
# it still simulated fl none the less.

from copy import deepcopy
import time
from torch import device
import os
import random
from model_functions import *
from model import ResNet18
from medmnist import ChestMNIST, dataset
from torchvision.transforms import v2
from torch.utils.data import DataLoader,Subset
import json

def fedavg(nodes_weights):
    global_dict = deepcopy(nodes_weights[0])
    for key in global_dict.keys():
        for i in range(1,len(nodes_weights)):
            global_dict[key] += nodes_weights[i][key]
        global_dict[key] = torch.div(global_dict[key],len(nodes_weights))
    return global_dict

def data_split(indices:list,peers:int,peer_id:int,test:bool=False):
        split_size = len(indices)//peers
        start = peer_id * split_size
        if test:
            # only take like 128 samples for testing purposes
            end = start + 4096
        else:
            if peers - 1 == peer_id:
                # last
                end = len(indices)
            else:
                end = (peer_id + 1) * split_size
        return indices[start:end]

def peer_split(datasets,peers,peer_id,random_seed):
    train_data,val_data = datasets

    train_len = len(train_data)
    val_len = len(val_data)
    train_indicies = list(range(train_len))
    val_indicies = list(range(val_len))

    rand = random.Random(random_seed)
    rand.shuffle(train_indicies)
    rand.shuffle(val_indicies)

    train_set = data_split(train_indicies,int(peers),int(peer_id))
    val_set = data_split(val_indicies,int(peers),int(peer_id))
    return train_set,val_set
    
def get_dataloaders(train_data,train_indices,val_data,val_indices,batch_size):
    train_dl = DataLoader(Subset(train_data,train_indices),batch_size=batch_size,shuffle=True)
    val_dl = DataLoader(Subset(val_data,val_indices),batch_size=batch_size,shuffle=True)
    return train_dl,val_dl

def fl_simulate(rounds,epochs,nodes,dataloaders,model:ResNet18):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    logs = [([[[],[]],[[],[]]]) for _ in range(len(nodes))]
    best_val_acc = [-1.0 for _ in range(len(nodes))]

    for i in range(rounds):
        for j in range(len(nodes)):
            print(f"[node_{j}] training")
            model.load_state_dict(nodes[j])
            loss_fn = torch.nn.BCEWithLogitsLoss()
            optimiser = torch.optim.Adam(model.parameters(), lr=5e-6) 

            for epoch in range(epochs):
                print(f"[node_{j}] Epoch {epoch+1}")
                train_dl,val_dl = dataloaders[j]
                train_loss,train_acc = train(train_dl,model,loss_fn,device,optimiser)
                val_loss,val_acc = test(val_dl,model,loss_fn,device)
                logs[j][0][0].append(train_loss)
                logs[j][0][1].append(train_acc)
                logs[j][1][0].append(val_loss)
                logs[j][1][1].append(val_acc)
                if best_val_acc[j] < val_acc:
                    os.makedirs(f"./results_sim/node_{j}", exist_ok=True)
                    torch.save(model.state_dict(),f"./results_sim/node_{j}/best_model.pth")
                    best_val_acc[j] = val_acc
            nodes[j] = deepcopy(model.state_dict())
            

        global_dict = fedavg(nodes)
        os.makedirs(f"./results_sim/global_models", exist_ok=True)
        torch.save(global_dict,f"./results_sim/global_models/global_model_{time.time()}.pth")
        nodes = [deepcopy(global_dict) for _ in range(len(nodes))]

    for j in range(len(nodes)):
        os.makedirs(f"./results_sim/node_{j}", exist_ok=True)
        file = open(f"./results_sim/node_{j}/log_{time.time()}.txt","w")
        file.write(json.dumps(logs[j]))
        file.close()
        
    

In [2]:

tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True)
])

train_data = ChestMNIST(split='train',transform=tf,download=True,root="./data")
val_data = ChestMNIST(split='val',transform=tf,download=True,root="./data")

classes = len(train_data.info["label"])
channels = train_data.info["n_channels"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNet18(channels,classes).to(device)

peers = 10

node_weights = [deepcopy(model.state_dict()) for _ in range(peers)]

peer_dataloaders = []

for i in range(peers):
    t_set,v_set = peer_split((train_data,val_data),peers,i,"random_seed_123")
    peer_dataloaders.append(get_dataloaders(train_data,t_set,val_data,v_set,32))

In [3]:
fl_simulate(rounds=20,epochs=5,nodes=node_weights,dataloaders=peer_dataloaders,model=model)

[node_0] training
[node_0] Epoch 1
Loss: 0.676591  [   32/ 7846]
Loss: 0.652481  [  832/ 7846]
Loss: 0.600057  [ 1632/ 7846]
Loss: 0.562145  [ 2432/ 7846]
Loss: 0.565835  [ 3232/ 7846]
Loss: 0.480636  [ 4032/ 7846]
Loss: 0.447394  [ 4832/ 7846]
Loss: 0.433559  [ 5632/ 7846]
Loss: 0.382475  [ 6432/ 7846]
Loss: 0.385561  [ 7232/ 7846]
Train Error: Accuracy: 79.8%, Average Loss: 0.501250
Test Error: Accuracy: 93.9%, Average Loss: 0.353847
[node_0] Epoch 2
Loss: 0.359678  [   32/ 7846]
Loss: 0.311709  [  832/ 7846]
Loss: 0.301943  [ 1632/ 7846]
Loss: 0.311191  [ 2432/ 7846]
Loss: 0.285125  [ 3232/ 7846]
Loss: 0.300975  [ 4032/ 7846]
Loss: 0.278780  [ 4832/ 7846]
Loss: 0.243370  [ 5632/ 7846]
Loss: 0.253431  [ 6432/ 7846]
Loss: 0.256002  [ 7232/ 7846]
Train Error: Accuracy: 94.7%, Average Loss: 0.285547
Test Error: Accuracy: 95.3%, Average Loss: 0.240829
[node_0] Epoch 3
Loss: 0.213505  [   32/ 7846]
Loss: 0.267506  [  832/ 7846]
Loss: 0.243410  [ 1632/ 7846]
Loss: 0.233749  [ 2432/ 7846]
L